# 15. Log-Mel CNN의 MP3 압축 강건성 평가

13단계에서 학습한 Log-Mel CNN을 재학습하지 않고 원본·128 kbps·64 kbps Test 오디오에 적용한다. Validation에서 정한 segment·track threshold도 그대로 사용한다.

MP3 중간 지점 seek 오차를 피하기 위해 각 파일을 처음부터 한 번 디코딩한 뒤 manifest의 시작점에서 10초를 자른다. 세 조건의 전체 성능과 generator별 ROC-AUC를 비교하고, 12단계 RBF-SVM 결과도 같은 표에 정리한다.

## 1. 경로 / 라이브러리 / Device 설정

In [1]:
from pathlib import Path
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "data").is_dir() and (p / "results").is_dir()),
    Path.cwd().resolve(),
)

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

ORIGINAL_LOGMEL_PATH = (
    PROJECT_ROOT / "data/processed/logmel/logmel_10s_float16.npy"
)
ORIGINAL_DONE_PATH = (
    PROJECT_ROOT / "data/processed/logmel/logmel_10s_done.npy"
)

COMPRESSED_ROOT = PROJECT_ROOT / "data/processed/mp3_robustness"

CNN_CHECKPOINT = PROJECT_ROOT / "checkpoints/cnn/logmel_cnn_best.pt"
CNN_THRESHOLD_PATH = PROJECT_ROOT / "results/cnn/cnn_thresholds.json"

SVM_MP3_METRICS = (
    PROJECT_ROOT / "results/mp3_robustness/mp3_robustness_metrics.csv"
)

CNN_MP3_CACHE_DIR = PROJECT_ROOT / "data/processed/logmel/mp3_robustness"
CNN_MP3_RESULT_DIR = PROJECT_ROOT / "results/cnn_mp3_robustness"

CNN_MP3_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CNN_MP3_RESULT_DIR.mkdir(parents=True, exist_ok=True)

SR = 24_000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)

N_FFT = 1024
HOP_LENGTH = 240
N_MELS = 128
FMAX = 12_000
N_FRAMES = 1 + TARGET_SAMPLES // HOP_LENGTH

BATCH_SIZE = 8
RANDOM_STATE = 42

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
print("Segment:", SEGMENT_PATH)
print("MP3 root:", COMPRESSED_ROOT)

PyTorch: 2.14.0
Device : mps
Segment: <PROJECT_ROOT>/data/metadata/segment_manifest_10s.csv
MP3 root: <PROJECT_ROOT>/data/processed/mp3_robustness


**실행 결과**

PyTorch 2.14.0을 불러왔고 연산 장치는 MPS로 설정되었다. segment manifest와 MP3 재인코딩 파일 경로도 함께 확인했다.

## 2. 재현성 설정

In [2]:
def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

**실행 내용**

Python·NumPy·PyTorch 난수 시드를 같은 값으로 맞추는 함수를 실행했다. 별도 수치 출력은 없다.

## 3. 입력 파일 / Test 구조 QC

In [3]:
required = [
    SEGMENT_PATH,
    ORIGINAL_LOGMEL_PATH,
    ORIGINAL_DONE_PATH,
    CNN_CHECKPOINT,
    CNN_THRESHOLD_PATH,
]

for p in required:
    print(p.name, "->", p.exists())

segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)
test_segments = segments[segments["split"] == "test"].copy()
test_segments["global_index"] = test_segments.index
test_segments = test_segments.reset_index(drop=True)

test_tracks = (
    test_segments
    .drop_duplicates("track_sample_id")
    .reset_index(drop=True)
)

original_cache = np.load(ORIGINAL_LOGMEL_PATH, mmap_mode="r")
original_done = np.load(ORIGINAL_DONE_PATH)

print("\nAll segment rows :", len(segments))
print("Test segments    :", len(test_segments))
print("Test tracks      :", len(test_tracks))
print("Original cache   :", original_cache.shape)
print("Original done    :", int(original_done.sum()), "/", len(original_done))

assert len(segments) == 10077
assert len(test_segments) == 1572
assert len(test_tracks) == 539
assert original_cache.shape == (10077, 128, 1001)
assert bool(original_done.all())

print("Test structure QC PASS: True")

segment_manifest_10s.csv -> True
logmel_10s_float16.npy -> True
logmel_10s_done.npy -> True
logmel_cnn_best.pt -> True
cnn_thresholds.json -> True

All segment rows : 10077
Test segments    : 1572
Test tracks      : 539
Original cache   : (10077, 128, 1001)
Original done    : 10077 / 10077
Test structure QC PASS: True


**실행 결과**

필요한 manifest, 원본 Log-Mel cache, 완료 mask, CNN checkpoint, threshold 파일이 모두 존재했다. 전체 10,077개 중 Test는 1,572개 segment·539개 track이며, 원본 cache `(10077, 128, 1001)`가 전부 완료된 상태다. 구조 검사값은 `True`다.

## 4. 12번에서 만든 MP3 파일 존재 여부 확인

In [4]:
def compressed_path(condition, track_sample_id):
    return (
        COMPRESSED_ROOT
        / condition
        / f"{track_sample_id}.mp3"
    )

rows = []

for condition in ["mp3_128", "mp3_64"]:
    exists_count = 0
    missing = []

    for track_id in test_tracks["track_sample_id"]:
        p = compressed_path(condition, track_id)
        if p.exists() and p.stat().st_size > 0:
            exists_count += 1
        else:
            missing.append(str(track_id))

    rows.append({
        "condition": condition,
        "expected_tracks": len(test_tracks),
        "existing_tracks": exists_count,
        "missing_tracks": len(missing),
    })

mp3_file_qc = pd.DataFrame(rows)
display(mp3_file_qc)

if (mp3_file_qc["missing_tracks"] > 0).any():
    raise FileNotFoundError(
        "12번 MP3 robustness 노트북에서 Test MP3 파일 생성 셀을 먼저 실행하세요."
    )

print("MP3 file QC PASS: True")

,condition,expected_tracks,existing_tracks,missing_tracks
0,mp3_128,539,539,0
1,mp3_64,539,539,0


MP3 file QC PASS: True


**실행 결과**

128 kbps와 64 kbps 폴더에서 Test 539개 track의 MP3 파일을 모두 찾았다. 두 조건의 누락 파일은 각각 0개다.

## 5. 13번 CNN 구조 및 checkpoint 로드

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(1, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(1)


model = LogMelCNN().to(DEVICE)

checkpoint = torch.load(
    CNN_CHECKPOINT,
    map_location=DEVICE,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())

with open(CNN_THRESHOLD_PATH, "r", encoding="utf-8") as f:
    thresholds = json.load(f)

SEGMENT_THRESHOLD = float(thresholds["segment_eer_threshold"])
TRACK_THRESHOLD = float(thresholds["track_eer_threshold"])

print("Best epoch       :", checkpoint["epoch"])
print("Total params     :", total_params)
print("Segment threshold:", SEGMENT_THRESHOLD)
print("Track threshold  :", TRACK_THRESHOLD)

assert total_params == 294321

Best epoch       : 18
Total params     : 294321
Segment threshold: 0.04316171258687973
Track threshold  : 0.09354320168495178


**실행 결과**

13단계와 같은 294,321-parameter CNN을 구성해 epoch 18 checkpoint를 불러왔다. 고정 threshold는 segment 0.04316, track 0.09354다.

## 6. MP3 segment → Log-Mel 함수

In [17]:
def load_full_compressed_audio(audio_path: Path):
    """
    MP3를 중간 지점에서 seek하지 않고,
    파일 처음부터 끝까지 한 번에 디코딩한다.
    """
    y, _ = librosa.load(
        audio_path,
        sr=SR,
        mono=True,
    )

    return np.asarray(y, dtype=np.float32)


def slice_segment_from_waveform(
    y,
    start_sec: float,
):
    start_sample = int(round(float(start_sec) * SR))
    end_sample = start_sample + TARGET_SAMPLES

    segment = y[start_sample:end_sample]

    if len(segment) < TARGET_SAMPLES:
        segment = np.pad(
            segment,
            (0, TARGET_SAMPLES - len(segment)),
            mode="constant",
        )

    elif len(segment) > TARGET_SAMPLES:
        segment = segment[:TARGET_SAMPLES]

    return np.asarray(segment, dtype=np.float32)


def waveform_to_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=FMAX,
        power=2.0,
        center=True,
    )

    logmel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    # [-80, 0] -> [-1, 1]
    logmel = (logmel + 80.0) / 40.0 - 1.0

    logmel = np.asarray(logmel, dtype=np.float32)

    if logmel.shape[1] < N_FRAMES:
        logmel = np.pad(
            logmel,
            ((0, 0), (0, N_FRAMES - logmel.shape[1])),
            mode="constant",
            constant_values=-1.0,
        )

    elif logmel.shape[1] > N_FRAMES:
        logmel = logmel[:, :N_FRAMES]

    assert logmel.shape == (N_MELS, N_FRAMES)

    return logmel

**실행 내용**

MP3를 처음부터 끝까지 한 번 디코딩하고 manifest의 시작점에서 240,000 samples를 자른 뒤, `(128, 1001)` Log-Mel로 바꾸는 함수를 정의했다. 이 셀에서는 파일을 처리하지 않았다.

## 7. 압축 조건별 Log-Mel cache 생성 — 128k / 64k

In [19]:
def build_compressed_logmel_cache(condition):
    cache_path = (
        CNN_MP3_CACHE_DIR
        / f"logmel_test_{condition}_fulldecode_float16.npy"
    )

    done_path = (
        CNN_MP3_CACHE_DIR
        / f"logmel_test_{condition}_fulldecode_done.npy"
    )

    cache = np.lib.format.open_memmap(
        cache_path,
        mode="w+",
        dtype=np.float16,
        shape=(len(test_segments), N_MELS, N_FRAMES),
    )

    done = np.zeros(
        len(test_segments),
        dtype=bool,
    )

    start_time = time.time()

    # 같은 track을 한 번만 전체 디코딩
    for track_num, (track_id, group) in enumerate(
        test_segments.groupby("track_sample_id"),
        start=1,
    ):
        audio_file = compressed_path(
            condition,
            track_id,
        )

        # MP3 전체를 처음부터 디코딩
        full_y = load_full_compressed_audio(audio_file)

        for idx, row in group.iterrows():

            # test_segments의 reset index 위치
            local_idx = test_segments.index[
                test_segments["segment_id"] == row["segment_id"]
            ][0]

            y_segment = slice_segment_from_waveform(
                full_y,
                row["start_sec"],
            )

            logmel = waveform_to_logmel(y_segment)

            cache[local_idx] = logmel.astype(np.float16)
            done[local_idx] = True

        if track_num % 50 == 0 or track_num == test_segments["track_sample_id"].nunique():
            cache.flush()
            np.save(done_path, done)

            elapsed = time.time() - start_time

            print(
                f"{condition}: "
                f"{track_num}/"
                f"{test_segments['track_sample_id'].nunique()} tracks "
                f"| {int(done.sum())}/{len(done)} segments "
                f"| elapsed {elapsed/60:.1f} min"
            )

    cache.flush()
    np.save(done_path, done)

    print(
        condition,
        "complete:",
        int(done.sum()),
        "/",
        len(done),
    )

    assert bool(done.all())

    return cache_path, done_path


compressed_cache_paths = {}

for condition in ["mp3_128", "mp3_64"]:

    print("\n" + "=" * 70)
    print("FULL-DECODE LOG-MEL CACHE:", condition)
    print("=" * 70)

    compressed_cache_paths[condition] = (
        build_compressed_logmel_cache(condition)
    )

print("Full-decode Compressed Log-Mel Cache QC PASS: True")


FULL-DECODE LOG-MEL CACHE: mp3_128
mp3_128: 50/539 tracks | 150/1572 segments | elapsed 0.1 min
mp3_128: 100/539 tracks | 300/1572 segments | elapsed 0.1 min
mp3_128: 150/539 tracks | 450/1572 segments | elapsed 0.2 min
mp3_128: 200/539 tracks | 600/1572 segments | elapsed 0.4 min
mp3_128: 250/539 tracks | 750/1572 segments | elapsed 0.5 min
mp3_128: 300/539 tracks | 879/1572 segments | elapsed 0.7 min
mp3_128: 350/539 tracks | 1005/1572 segments | elapsed 0.8 min
mp3_128: 400/539 tracks | 1155/1572 segments | elapsed 0.9 min
mp3_128: 450/539 tracks | 1305/1572 segments | elapsed 1.1 min
mp3_128: 500/539 tracks | 1455/1572 segments | elapsed 1.3 min
mp3_128: 539/539 tracks | 1572/1572 segments | elapsed 1.3 min
mp3_128 complete: 1572 / 1572

FULL-DECODE LOG-MEL CACHE: mp3_64
mp3_64: 50/539 tracks | 150/1572 segments | elapsed 0.1 min
mp3_64: 100/539 tracks | 300/1572 segments | elapsed 0.1 min
mp3_64: 150/539 tracks | 450/1572 segments | elapsed 0.2 min
mp3_64: 200/539 tracks | 600/15

**실행 결과**

두 압축 조건에서 539개 track의 전체 디코딩과 1,572개 segment 변환을 마쳤다. 128 kbps는 약 1.3분, 64 kbps는 약 1.2분이 걸렸고 두 cache의 완료 수는 모두 1,572/1,572다.

## 8. 평가 Dataset / DataLoader

In [20]:
class ConditionDataset(Dataset):
    def __init__(
        self,
        metadata,
        cache,
        cache_indices,
    ):
        self.metadata = metadata.reset_index(drop=True)
        self.cache = cache
        self.cache_indices = np.asarray(cache_indices, dtype=int)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, i):
        cache_idx = int(self.cache_indices[i])

        x = np.asarray(
            self.cache[cache_idx],
            dtype=np.float32,
        )

        x = torch.from_numpy(x).unsqueeze(0)

        y = torch.tensor(
            float(self.metadata.iloc[i]["label_id"]),
            dtype=torch.float32,
        )

        return x, y, i


def make_condition_loader(condition):
    if condition == "original":
        cache = original_cache
        indices = test_segments["global_index"].to_numpy()
    else:
        cache_path, _ = compressed_cache_paths[condition]
        cache = np.load(cache_path, mmap_mode="r")
        indices = np.arange(len(test_segments))

    ds = ConditionDataset(
        test_segments,
        cache,
        indices,
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    return ds, loader

**실행 내용**

원본 또는 압축 Log-Mel cache를 읽어 순서가 고정된 Test DataLoader를 만드는 dataset과 loader 함수를 정의했다.

## 9. 예측 / 평가 함수

In [21]:
@torch.no_grad()
def predict_loader(model, loader):
    ys = []
    scores = []
    indices = []

    model.eval()

    for x, y, idx in loader:
        x = x.to(DEVICE)

        logits = model(x)
        prob = torch.sigmoid(logits)

        ys.append(y.numpy())
        scores.append(
            prob.detach().cpu().numpy()
        )
        indices.append(idx.numpy())

    return (
        np.concatenate(ys),
        np.concatenate(scores),
        np.concatenate(indices),
    )


def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)

    fpr = fpr[valid]
    fnr = fnr[valid]
    thresholds = thresholds[valid]

    idx = np.argmin(np.abs(fpr - fnr))

    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    eer_info = find_eer_threshold(
        y_true,
        scores,
    )

    return {
        "roc_auc": float(
            roc_auc_score(y_true, scores)
        ),
        "pr_auc": float(
            average_precision_score(y_true, scores)
        ),
        "eer": float(eer_info["eer"]),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "real_fpr": float(fp / (fp + tn)),
        "fake_miss_rate": float(fn / (fn + tp)),
        "threshold_used": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_track_scores(metadata, scores):
    temp = metadata[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()

    temp["score"] = scores

    return (
        temp
        .groupby("track_sample_id", as_index=False)
        .agg(
            original_audio=("original_audio", "first"),
            label=("label", "first"),
            label_id=("label_id", "first"),
            genre=("genre", "first"),
            generator=("generator", "first"),
            split=("split", "first"),
            segment_count=("score", "size"),
            score=("score", "mean"),
        )
    )

**실행 내용**

CNN 확률 예측, EER와 고정-threshold 지표 계산, segment score의 track 평균 집계를 위한 함수를 정의했다.

## 10. Original / MP3 128 / MP3 64 CNN 평가

In [22]:
condition_results = []
track_outputs = {}
segment_outputs = {}

for condition in ["original", "mp3_128", "mp3_64"]:
    print("\n" + "=" * 70)
    print("EVALUATION:", condition)
    print("=" * 70)

    _, loader = make_condition_loader(condition)

    y, scores, local_indices = predict_loader(
        model,
        loader,
    )

    meta_ordered = (
        test_segments.iloc[local_indices]
        .reset_index(drop=True)
    )

    segment_metrics = evaluate_scores(
        y,
        scores,
        SEGMENT_THRESHOLD,
    )

    track_df = make_track_scores(
        meta_ordered,
        scores,
    )

    track_metrics = evaluate_scores(
        track_df["label_id"],
        track_df["score"],
        TRACK_THRESHOLD,
    )

    condition_results.append({
        "condition": condition,
        "level": "segment",
        **segment_metrics,
    })

    condition_results.append({
        "condition": condition,
        "level": "track",
        **track_metrics,
    })

    segment_outputs[condition] = (
        meta_ordered.copy(),
        scores.copy(),
    )
    track_outputs[condition] = (
        track_df.copy()
    )

cnn_mp3_metrics = pd.DataFrame(condition_results)

display(
    cnn_mp3_metrics[
        [
            "condition",
            "level",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ].round(4)
)


EVALUATION: original

EVALUATION: mp3_128

EVALUATION: mp3_64


,condition,level,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate,threshold_used
0,original,segment,0.9520,0.1349,0.8791,0.7795,0.1556,0.0863,0.0432
1,original,track,0.9772,0.1042,0.9111,0.8535,0.1333,0.0445,0.0935
2,mp3_128,segment,0.8454,0.2365,0.6871,0.6718,0.5556,0.0703,0.0432
3,mp3_128,track,0.8954,0.1830,0.7626,0.7765,0.4444,0.0304,0.0935
4,mp3_64,segment,0.7586,0.3169,0.6794,0.4974,0.2444,0.3967,0.0432
5,mp3_64,track,0.8256,0.2892,0.7179,0.5430,0.2444,0.3198,0.0935


**실행 결과**

원본·128 kbps·64 kbps의 segment/track 결과 6행을 계산했다. Track ROC-AUC는 0.9772→0.8954→0.8256, EER은 0.1042→0.1830→0.2892였다.

## 11. Track-level 결과 및 성능 변화

In [23]:
cnn_track = (
    cnn_mp3_metrics[
        cnn_mp3_metrics["level"] == "track"
    ]
    .copy()
)

order = pd.CategoricalDtype(
    ["original", "mp3_128", "mp3_64"],
    ordered=True,
)

cnn_track["condition"] = (
    cnn_track["condition"].astype(order)
)

cnn_track = cnn_track.sort_values("condition")

display(
    cnn_track[
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].round(4)
)

orig = cnn_track[
    cnn_track["condition"] == "original"
].iloc[0]

drop_rows = []

for _, r in cnn_track.iterrows():
    drop_rows.append({
        "condition": r["condition"],
        "roc_auc_drop_vs_original":
            float(orig["roc_auc"] - r["roc_auc"]),
        "eer_change_vs_original":
            float(r["eer"] - orig["eer"]),
        "real_fpr_change_vs_original":
            float(r["real_fpr"] - orig["real_fpr"]),
        "fake_miss_change_vs_original":
            float(r["fake_miss_rate"] - orig["fake_miss_rate"]),
    })

compression_drop = pd.DataFrame(drop_rows)

print("Compression change vs Original")
display(compression_drop.round(4))

,condition,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate
1,original,0.9772,0.1042,0.9111,0.8535,0.1333,0.0445
3,mp3_128,0.8954,0.1830,0.7626,0.7765,0.4444,0.0304
5,mp3_64,0.8256,0.2892,0.7179,0.5430,0.2444,0.3198


Compression change vs Original


,condition,roc_auc_drop_vs_original,eer_change_vs_original,real_fpr_change_vs_original,fake_miss_change_vs_original
0,original,0.0000,0.0000,0.0000,0.0000
1,mp3_128,0.0818,0.0789,0.3111,-0.0142
2,mp3_64,0.1516,0.1850,0.1111,0.2753


**실행 결과**

Track 기준 원본/128 kbps/64 kbps의 balanced accuracy는 0.9111/0.7626/0.7179다. 원본 대비 ROC-AUC 감소는 128 kbps 0.0818, 64 kbps 0.1516이며, 64 kbps의 FAKE miss rate는 0.3198이다.

## 12. 13번 Original 결과와 일치하는지 QC

In [24]:
expected_original = {
    "roc_auc": 0.9772,
    "eer": 0.1042,
    "balanced_accuracy": 0.9111,
    "macro_f1": 0.8535,
    "real_fpr": 0.1333,
    "fake_miss_rate": 0.0445,
}

original_row = cnn_track[
    cnn_track["condition"] == "original"
].iloc[0]

for k, expected in expected_original.items():
    print(
        k,
        "current =", round(float(original_row[k]), 4),
        "| expected ≈", expected,
    )

print(
    "Original result consistency:",
    all(
        abs(float(original_row[k]) - v) < 0.002
        for k, v in expected_original.items()
    )
)

roc_auc current = 0.9772 | expected ≈ 0.9772
eer current = 0.1042 | expected ≈ 0.1042
balanced_accuracy current = 0.9111 | expected ≈ 0.9111
macro_f1 current = 0.8535 | expected ≈ 0.8535
real_fpr current = 0.1333 | expected ≈ 0.1333
fake_miss_rate current = 0.0445 | expected ≈ 0.0445
Original result consistency: True


**실행 결과**

재계산한 원본 track 지표 여섯 개가 13단계 결과와 소수 넷째 자리까지 일치했다. 일관성 검사값은 `True`다.

## 13. Generator별 Track ROC-AUC

In [25]:
def evaluate_generator_track(track_df):
    real_df = track_df[
        track_df["label"] == "REAL"
    ]

    fake_df = track_df[
        track_df["label"] == "FAKE"
    ]

    rows = []

    for generator in sorted(
        fake_df["generator"].dropna().unique()
    ):
        subgroup = pd.concat(
            [
                real_df,
                fake_df[
                    fake_df["generator"] == generator
                ],
            ],
            ignore_index=True,
        )

        rows.append({
            "generator": generator,
            "n_fake": int(
                (subgroup["label"] == "FAKE").sum()
            ),
            "roc_auc": float(
                roc_auc_score(
                    subgroup["label_id"],
                    subgroup["score"],
                )
            ),
        })

    return pd.DataFrame(rows)


generator_rows = []

for condition in ["original", "mp3_128", "mp3_64"]:
    temp = evaluate_generator_track(
        track_outputs[condition]
    )
    temp["condition"] = condition
    generator_rows.append(temp)

generator_metrics = pd.concat(
    generator_rows,
    ignore_index=True,
)

generator_auc_table = generator_metrics.pivot(
    index="generator",
    columns="condition",
    values="roc_auc",
)

display(generator_auc_table.round(4))

generator_auc_table["drop_64_vs_original"] = (
    generator_auc_table["original"]
    - generator_auc_table["mp3_64"]
)

print("\nLargest 64k AUC drops")
display(
    generator_auc_table
    .sort_values("drop_64_vs_original", ascending=False)
    .head(12)
    .round(4)
)

condition,mp3_128,mp3_64,original
generator,,,
acestep,0.9165,0.8504,0.9881
audioldm,0.9936,0.9956,1.0000
brev,0.8519,0.7639,0.9736
diffrhythm,0.8884,0.6861,0.9901
elevenlabs,0.9324,0.7917,0.9606
mubert,0.7417,0.6648,0.9241
musicgen,0.8602,0.9437,0.9748
producer,0.9147,0.8187,0.9822
songgen,0.9985,0.9980,1.0000



Largest 64k AUC drops


condition,mp3_128,mp3_64,original,drop_64_vs_original
generator,,,,
diffrhythm,0.8884,0.6861,0.9901,0.3040
suno,0.8190,0.6935,0.9569,0.2634
mubert,0.7417,0.6648,0.9241,0.2593
brev,0.8519,0.7639,0.9736,0.2097
udio,0.8481,0.7694,0.9671,0.1977
elevenlabs,0.9324,0.7917,0.9606,0.1690
producer,0.9147,0.8187,0.9822,0.1636
acestep,0.9165,0.8504,0.9881,0.1378
stableaudio,0.9479,0.9179,0.9983,0.0803


**실행 결과**

64 kbps에서 generator별 ROC-AUC 감소가 가장 큰 경우는 DiffRhythm 0.3040, Suno 0.2634, Mubert 0.2593이었다. AudioLDM과 SongGen의 감소는 각각 0.0044와 0.0020이었다.

## 14. RBF-SVM vs CNN — MP3 robustness 비교

In [26]:
if SVM_MP3_METRICS.exists():
    svm_all = pd.read_csv(SVM_MP3_METRICS)

    svm_track = svm_all[
        (svm_all["model"] == "RBF-SVM")
        & (svm_all["level"] == "track")
    ][
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    svm_track["model"] = "RBF-SVM"

    cnn_compare = cnn_track[
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    cnn_compare["model"] = "LogMelCNN"

    model_comparison = pd.concat(
        [svm_track, cnn_compare],
        ignore_index=True,
    )

    display(
        model_comparison[
            [
                "model",
                "condition",
                "roc_auc",
                "eer",
                "balanced_accuracy",
                "macro_f1",
                "real_fpr",
                "fake_miss_rate",
            ]
        ].round(4)
    )
else:
    print(
        "12번 SVM MP3 metrics 파일이 없어 "
        "CNN 결과만 저장합니다."
    )
    model_comparison = pd.DataFrame()

,model,condition,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate
0,RBF-SVM,original,0.9644,0.1395,0.8696,0.7502,0.1556,0.1053
1,RBF-SVM,mp3_128,0.9536,0.1325,0.8676,0.7692,0.1778,0.0870
2,RBF-SVM,mp3_64,0.9062,0.1688,0.8494,0.7437,0.2000,0.1012
3,LogMelCNN,original,0.9772,0.1042,0.9111,0.8535,0.1333,0.0445
4,LogMelCNN,mp3_128,0.8954,0.1830,0.7626,0.7765,0.4444,0.0304
5,LogMelCNN,mp3_64,0.8256,0.2892,0.7179,0.5430,0.2444,0.3198


**실행 결과**

같은 Test 조건에서 RBF-SVM과 CNN을 비교했다. Track ROC-AUC는 원본에서 CNN 0.9772가 SVM 0.9644보다 높았지만, 128 kbps와 64 kbps에서는 SVM 0.9536/0.9062가 CNN 0.8954/0.8256보다 높았다.

## 15. 결과 저장

In [27]:
cnn_mp3_metrics.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_mp3_robustness_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

compression_drop.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_track_compression_change.csv",
    index=False,
    encoding="utf-8-sig",
)

generator_metrics.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_generator_compression_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

generator_auc_table.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_generator_compression_auc_table.csv",
    encoding="utf-8-sig",
)

if len(model_comparison):
    model_comparison.to_csv(
        CNN_MP3_RESULT_DIR / "svm_vs_cnn_mp3_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Saved to:", CNN_MP3_RESULT_DIR)

Saved to: <PROJECT_ROOT>/results/cnn_mp3_robustness


**실행 결과**

CNN 압축 지표, 원본 대비 변화량, generator별 압축 지표와 SVM 비교표를 `results/cnn_mp3_robustness`에 저장했다.

## 16. 최종 QC

In [28]:
cache_128 = np.load(
    compressed_cache_paths["mp3_128"][0],
    mmap_mode="r",
)
cache_64 = np.load(
    compressed_cache_paths["mp3_64"][0],
    mmap_mode="r",
)

done_128 = np.load(
    compressed_cache_paths["mp3_128"][1]
)
done_64 = np.load(
    compressed_cache_paths["mp3_64"][1]
)

qc = pd.DataFrame({
    "check": [
        "test_segments",
        "test_tracks",
        "mp3_128_tracks",
        "mp3_64_tracks",
        "mp3_128_cache_rows",
        "mp3_64_cache_rows",
        "mp3_128_cache_complete",
        "mp3_64_cache_complete",
        "model_params",
        "result_rows",
        "track_conditions",
        "original_consistency",
    ],
    "value": [
        len(test_segments),
        len(test_tracks),
        int(mp3_file_qc.loc[
            mp3_file_qc["condition"] == "mp3_128",
            "existing_tracks"
        ].iloc[0]),
        int(mp3_file_qc.loc[
            mp3_file_qc["condition"] == "mp3_64",
            "existing_tracks"
        ].iloc[0]),
        cache_128.shape[0],
        cache_64.shape[0],
        bool(done_128.all()),
        bool(done_64.all()),
        total_params,
        len(cnn_mp3_metrics),
        cnn_track["condition"].nunique(),
        all(
            abs(float(original_row[k]) - v) < 0.002
            for k, v in expected_original.items()
        ),
    ],
})

display(qc)

core_qc_pass = (
    len(test_segments) == 1572
    and len(test_tracks) == 539
    and (mp3_file_qc["existing_tracks"] == 539).all()
    and cache_128.shape == (1572, 128, 1001)
    and cache_64.shape == (1572, 128, 1001)
    and bool(done_128.all())
    and bool(done_64.all())
    and total_params == 294321
    and len(cnn_mp3_metrics) == 6
    and cnn_track["condition"].nunique() == 3
    and all(
        abs(float(original_row[k]) - v) < 0.002
        for k, v in expected_original.items()
    )
)

print("===== FINAL RESULT =====")
print("CNN MP3 Robustness Core QC PASS:", core_qc_pass)

,check,value
0,test_segments,1572
1,test_tracks,539
2,mp3_128_tracks,539
3,mp3_64_tracks,539
4,mp3_128_cache_rows,1572
5,mp3_64_cache_rows,1572
6,mp3_128_cache_complete,True
7,mp3_64_cache_complete,True
8,model_params,294321
9,result_rows,6


===== FINAL RESULT =====
CNN MP3 Robustness Core QC PASS: True


**실행 결과**

Test 1,572개 segment·539개 track, 두 조건의 539개 MP3와 완성된 `(1572, 128, 1001)` cache를 확인했다. 모델 parameter는 294,321개, 결과는 6행·3조건이며 원본 일관성도 `True`다. 최종 출력은 `CNN MP3 Robustness Core QC PASS: True`다.

## 결과 정리

| 조건 | Track ROC-AUC | EER | Balanced accuracy | REAL FPR | FAKE miss rate |
|---|---:|---:|---:|---:|---:|
| Original | 0.9772 | 0.1042 | 0.9111 | 0.1333 | 0.0445 |
| MP3 128 kbps | 0.8954 | 0.1830 | 0.7626 | 0.4444 | 0.0304 |
| MP3 64 kbps | 0.8256 | 0.2892 | 0.7179 | 0.2444 | 0.3198 |

원본 대비 ROC-AUC 감소는 128 kbps에서 0.0818, 64 kbps에서 0.1516이었다. 같은 조건의 RBF-SVM ROC-AUC는 0.9644/0.9536/0.9062다. 최종 검사 결과는 `CNN MP3 Robustness Core QC PASS: True`다.